<h1 style="color:green;font-size:22px;">Function 4 - Black-Box Optimisation</h1>
<h1 style="color:#0000CD;font-size:19px;"">Introduction and illustrative analogy</h1>

**Function 4** is a four-dimensional black-box objective over the bounded domain $[0,1]^4$. Its analytical form and physical interpretation are unknown. The objective is to identify high-value input configurations under a limited sequential-query budget.

As an illustrative analogy, the four inputs may be viewed as tunable parameters in an expensive operational model. This analogy is motivational only: the optimisation below relies solely on observed input-output pairs and does not assume that the hidden function represents warehouse placement or any specific business process. The goal is to maximise **Function 4**.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as s
import seaborn as sns
import warnings

from itertools import combinations
from mpl_toolkits.mplot3d import Axes3D
from sklearn.exceptions import ConvergenceWarning

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<h1 style="color:#0000CD;font-size:19px;"">Week 1</h1>

**1.1 - Extraction of Initial Data**

In [4]:
inputs = np.load('Initial Data/function_4/initial_inputs.npy')
outputs = np.load('Initial Data/function_4/initial_outputs.npy')
print(inputs.shape, outputs.shape)

(30, 4) (30,)


In [5]:
data = pd.DataFrame(inputs, columns=['x1','x2','x3','x4'])
data['y'] = outputs
display(data)

,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [6]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-4.025542281908162 28.600117934054293


**1.2 - Optimisation**

In [5]:
# # Full 4D tensor candidate grid
x1 = np.linspace(0,1,50)
x2 = np.linspace(0,1,50)
x3 = np.linspace(0,1,50)
x4 = np.linspace(0,1,50)

xx1, xx2, xx3, xx4 = np.meshgrid(x1, x2, x3, x4)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel()])
del xx1, xx2, xx3, xx4

# Predict GP mean and uncertainty
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound
kappa = 1.5
ucb = y_pred + kappa * sigma
index_max = np.argmax(ucb)
next_point = np.array(X_grid[index_max], float)
print(f"Next point (with GP+UCB):{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}")



Next point (with GP+UCB):0.428571-0.448980-0.367347-0.448980


<h1 style="color:#0000CD;font-size:19px;"">Week 2</h1>

**2.1 -  Previous Week's Query Result**

In [7]:
# New Query Point
x_new = np.array([[0.428571, 0.448980, 0.367347,0.448980]])
y_new = -0.17915169292686128

def add_QueriedPoint(data, x_new, y_new):

    inputs = data[['x1', 'x2','x3', 'x4']].to_numpy()
    outputs = data['y'].to_numpy()
    
    # Checks if New Points is already included in data
    exists = False
    for i in range(inputs.shape[0]):
        if np.allclose(inputs[i], x_new) and np.isclose(outputs[i], y_new):
            exists = True
            break

    # Only adds if it doesn't exist already 
    if not exists:
        inputs = np.vstack([inputs, x_new])
        outputs = np.append(outputs, y_new)
        data = pd.DataFrame(inputs, columns=['x1', 'x2','x3', 'x4']).assign(y=outputs)
    
        print("Point added!")
    else:
        print("Point already exists, skipping addition.")
    return data

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [8]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.17915169292686128 32.4465085230356


**Deduction:**

The first UCB query returned −0.179152, improving the initial incumbent from −4.025542. Because only one sequential observation had been collected, the optimisation remained UCB-led rather than switching immediately to pure exploitation.

**2.2 - Next Point Query**

In [8]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 1.5
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.40816327 0.44897959 0.42857143 0.42857143]


<h1 style="color:#0000CD;font-size:19px;"">Week 3</h1>

**3.1 -  Previous Week's Query Result**

In [9]:
# New Query Point
x_new = np.array([[0.408163, 0.448980, 0.428571,0.428571]])
y_new = 0.15826768403414748

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [10]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

0.15826768403414748 32.783927899996606


**Deduction:**

The second UCB query at [0.408163, 0.448980, 0.428571, 0.428571] returned y=0.158268, establishing a new best observed value. UCB was retained for another round to investigate the local neighbourhood while preserving uncertainty-driven exploration.

**3.2 - Next Point Query**

In [11]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.40816327 0.40816327 0.44897959 0.44897959]


<h1 style="color:#0000CD;font-size:19px;"">Week 4</h1>

**4.1 -  Previous Week's Query Result**

In [11]:
# New Query Point
x_new = np.array([[0.408163, 0.408163, 0.448980,0.448980]])
y_new = -0.24276654679536902

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [12]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

0.15826768403414748 32.783927899996606


**4.2 - Next Point Query**

In [14]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 3
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.40816327 0.48979592 0.42857143 0.40816327]


<h1 style="color:#0000CD;font-size:19px;"">Week 5</h1>

**5.1 -  Previous Week's Query Result**

In [13]:
# New Query Point
x_new = np.array([[0.408163, 0.489796, 0.428571, 0.408163]])
y_new = -0.9807108960285293

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [14]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

0.15826768403414748 32.783927899996606


**5.2 - Next Point Query (Adaptive UCB vs EI Comparison)**

The UCB candidate was selected to retain limited exploration around the incumbent region.

In [17]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
## We removed the EI switch
#if abs(max_ucb - f_max_obs) <= eps:
    
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(5)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.42857143 0.42857143 0.36734694 0.44897959]
Next point (UCB): [0.42857143 0.42857143 0.3877551  0.44897959]


<h1 style="color:#0000CD;font-size:19px;"">Week 6</h1>

**6.1 -  Previous Week's Query Result**

In [15]:
# New Query Point
x_new = np.array([[0.428571, 0.428571, 0.387755, 0.448980]])
y_new = 0.08563978005291917

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [16]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

0.15826768403414748 32.783927899996606


**6.2 - Next Point Query (Adaptive UCB vs EI Comparison)**

The EI candidate was selected for tighter local refinement.

In [20]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(6)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.42857143 0.40816327 0.3877551  0.44897959]
Next point (UCB): [0.42857143 0.42857143 0.3877551  0.44897959]


<h1 style="color:#0000CD;font-size:19px;"">Week 7</h1>

**7.1 -  Previous Week's Query Result**

In [17]:
# New Query Point
x_new = np.array([[0.428571, 0.408163, 0.387755, 0.448980]])
y_new = 0.037359140559438675

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [18]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)


0.15826768403414748 32.783927899996606


**7.2 - Next Point Query (EI Refinement with UCB Benchmark)**

EI and UCB identified the same candidate, so the selection was unambiguous. Let us increase resolution for refinement.

**Note:** The tensor-grid search reproduces the candidate-generation approach used during the challenge. It is computationally intensive in four dimensions; a lower-memory quasi-random or batched candidate search is reserved for the subsequent code-refactoring version.

In [23]:
# # Full 4D tensor candidate grid
x1 = np.linspace(0,1,75)
x2 = np.linspace(0,1,75)
x3 = np.linspace(0,1,75)
x4 = np.linspace(0,1,75)

xx1, xx2, xx3, xx4 = np.meshgrid(x1, x2, x3, x4)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel()])
del xx1, xx2, xx3, xx4

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(7)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.41891892 0.41891892 0.37837838 0.44594595]
Next point (UCB): [0.41891892 0.41891892 0.37837838 0.44594595]


<h1 style="color:#0000CD;font-size:19px;"">Week 8</h1>

**8.1 -  Previous Week's Query Result**

In [19]:
# New Query Point
x_new = np.array([[0.418919, 0.418919, 0.378378, 0.445946]])
y_new = 0.2557378049254351

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**8.2 - Next Point Query (EI Refinement with UCB Benchmark)**

EI and UCB proposed nearby candidates. Because the optimisation had already transitioned to EI-led local refinement, the EI candidate was retained.

In [25]:
# Reuse of the  Full 4D tensor candidate grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(8)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.24324324 0.41891892 0.37837838 0.41891892]
Next point (UCB): [0.21621622 0.41891892 0.37837838 0.43243243]


<h1 style="color:#0000CD;font-size:19px;"">Week 9</h1>

**9.1 -  Previous Week's Query Result**

In [20]:
# New Query Point
x_new = np.array([[0.243243, 0.418919, 0.378378, 0.418919]])
y_new = -1.7103003939299373

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**9.2 - Next Point Query (EI Refinement with UCB Benchmark)**

In [27]:
# Full 4D tensor candidate grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(9)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.37837838 0.37837838 0.39189189 0.35135135]
Next point (UCB): [0.37837838 0.36486486 0.39189189 0.32432432]


<h1 style="color:#0000CD;font-size:19px;"">Week 10</h1>

**10.1 -  Previous Week's Query Result**

In [21]:
# New Query Point
x_new = np.array([[0.378378, 0.378378, 0.391892, 0.351351]])
y_new = 0.3039447812105718

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**10.2 - Next Point Query (EI with UCB Benchmark)**

In [29]:
# Full 4D tensor candidate grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(10)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.37837838 0.41891892 0.39189189 0.41891892]
Next point (UCB): [0.36486486 0.41891892 0.39189189 0.41891892]


<h1 style="color:#0000CD;font-size:19px;"">Week 11</h1>

**11.1 -  Previous Week's Query Result**

In [22]:
# New Query Point
x_new = np.array([[0.378378, 0.418919, 0.391892, 0.418919]])
y_new = 0.5218920131442286

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**11.2 - Next Point Query (EI with UCB Benchmark)**

In [31]:
# Full 4D tensor candidate grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(11)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.41891892 0.40540541 0.36486486 0.43243243]
Next point (UCB): [0.41891892 0.40540541 0.36486486 0.43243243]


<h1 style="color:#0000CD;font-size:19px;"">Week 12</h1>

**12.1 -  Previous Week's Query Result**

In [23]:
# New Query Point
x_new = np.array([[0.418919, 0.405405, 0.364865, 0.432432]])
y_new = 0.5851855078535029

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**12.2 - Next Point Query (EI with UCB Benchmark)**

In [33]:
# Reuse of Full 4D tensor candidate grid X_grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)
      
# Pure EI
f_max_obs = max(outputs)
improvement = y_pred - f_max_obs
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma == 0.0] = 0.0  # avoid divide by zero
index_ei = np.argmax(ei)
next_point_ei = X_grid[index_ei]
print(f"Next point (EI): {next_point_ei}")
    
# Adaptive UCB
kappa = 2/ np.sqrt(12)
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]
print(f"Next point (UCB): {next_point_ucb}")

Next point (EI): [0.47297297 0.33783784 0.17567568 0.31081081]
Next point (UCB): [0.45945946 0.33783784 0.17567568 0.31081081]


<h1 style="color:#0000CD;font-size:19px;"">Week 13</h1>

**13.1 -  Previous Week's Query Result**

In [24]:
# New Query Point
x_new = np.array([[0.472973, 0.337838, 0.175676, 0.310811]])
y_new = -4.573956476256928

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


**13.2 - Next Point Query (EI)**

The previous EI proposal returned y=−4.573956 and did not improve the incumbent. For the final query, the search was therefore restricted to a local region around the current best observation. A scrambled Sobol design was used to generate candidates within ±0.12 of the incumbent, while candidates within Euclidean distance 0.05 of an existing observation were excluded. The surrogate was also changed to a Matérn-5/2 Gaussian process with optimised dimension-specific length scales. This final-stage change is intended as a more robust local refinement step rather than a continuation of the earlier tensor-grid search.

In [23]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove candidates too close to existing observations to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.00
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)


Current best point: [0.418919 0.405405 0.364865 0.432432]
Current best value: 0.5851855078535029
Next point (EI): [0.39581797 0.36666935 0.38616425 0.41845613]


<h1 style="color:#0000CD;font-size:19px;"">Final Result</h1>

**14.1 -  Previous Week's Query Result**

In [25]:
# Queried Point result
x_new = np.array([[0.395818, 0.366669, 0.386164, 0.418456]])
y_new = 0.4355106831799387

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2', 'x3', 'x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


The initial design contained 30 observations, with an initial incumbent of y=−4.025542. Across 13 sequential queries, the optimisation improved the best observed value to y=0.585186 at approximately [0.418919, 0.405405, 0.364865, 0.432432].

The final local EI query at [0.395818, 0.366669, 0.386164, 0.418456] returned y=0.435511. This was a relatively strong observation but did not exceed the incumbent. The final best observed value therefore remained 0.585186.

Overall, the sequential search identified a compact high-value region around x1≈0.38–0.42, x2≈0.38–0.42, x3≈0.36–0.39, and x4≈0.42–0.45. The isolated poor results also illustrate that local acquisition proposals remained sensitive to the surrogate and candidate-search design.